In [1]:
# Import the libraries

%pip install yfinance 
%pip install plotly

import yfinance as yf

import pandas as pd 
import numpy as np 

import plotly.graph_objects as go 
import plotly.express as px 
#import plotly.io as pio



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
# Download the stock data from yfinance
import yfinance as yf
# Define the stock ticker and the date range
tickers = ['AAPL']
start_date = '2020-01-01'
end_date = '2026-01-01'

# Download the stock data
stocks = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)

# Arrange the data so that 'Close' price can directly be used for plotting and calculations
stocks.columns = stocks.columns.droplevel(1)  # Drop the ticker level from columns

# reset the index to make 'Date' a column
stocks.reset_index(inplace=True)
# Drop the date column
stocks.drop(columns=['Date'], inplace=True)

stocks.head()

[*********************100%***********************]  1 of 1 completed

Price,Close,High,Low,Open,Volume
0,72.400513,72.460776,71.156674,71.409778,135480400
1,71.696648,72.455966,71.472469,71.629153,146322800
2,72.267921,72.306491,70.568495,70.819193,118387200
3,71.928040,72.533080,71.708680,72.277563,108872000
4,73.085121,73.386438,71.631567,71.631567,132079200


In [17]:
### Add Bollinger Bands to the stock data
# Calculate the 20-day moving average and standard deviation


def Bollinger_Bands(df, window=20, num_std_dev=2):
    df['MA20'] = df['Close'].rolling(window=window).mean()
    df['STD20'] = df['Close'].rolling(window=window).std()
    df['UpperBand'] = df['MA20'] + (df['STD20'] * num_std_dev)
    df['LowerBand'] = df['MA20'] - (df['STD20'] * num_std_dev)
    return df
stocks = Bollinger_Bands(stocks)  
#stocks.dropna(inplace=True)
#.reset_index(inplace=True)  # Drop rows with NaN values resulting from rolling calculations

stocks.head()

# Plot the stock price and Bollinger Bands
#pio.renderers.default = "browser"

fig = go.Figure().update_layout(width=1000, height=600, title='AAPL Stock Price with Bollinger Bands', xaxis_title='Date', yaxis_title='Price (USD)')
fig.add_trace(go.Scatter(x=stocks.index, y=stocks['Close'], mode='lines', name='Close Price'))

fig.add_trace(go.Scatter(x=stocks.index, 
                         y=stocks['UpperBand'], 
                         mode = 'lines', 
                         name='Upper Bollinger Band', 
                         line=dict(color='red', dash='dot')))

fig.add_trace(go.Scatter(x=stocks.index, 
                         y=stocks['LowerBand'], 
                         mode = 'lines', 
                         name='Lower Bollinger Band', 
                         line=dict(color='red', dash='dot')))
fig.show()


In [ ]:
# Get 20 day, 50 day, and 200 day moving averages
stocks['MA50'] = stocks['Close'].rolling(window=50).mean()
stocks['MA200'] = stocks['Close'].rolling(window=200).mean()

# Get the lag features for the closing price
stocks['Lag1'] = stocks['Close'].shift(1)
stocks['Lag2'] = stocks['Close'].shift(2)

# Capture volume spikes
stocks['VolumeSpike'] = stocks['Volume'] > (stocks['Volume'].rolling(window=20).mean() + 2*stocks['Volume'].rolling(window=20).std())

# 
###stocks.dropna(inplace=True)
stocks.head()


Price,Close,High,Low,Open,Volume,MA50,MA200,Lag1,Lag2,VolumeSpike
0,72.400513,72.460776,71.156674,71.409778,135480400,NaN,NaN,NaN,NaN,False
1,71.696648,72.455966,71.472469,71.629153,146322800,NaN,NaN,72.400513,NaN,False
2,72.267921,72.306491,70.568495,70.819193,118387200,NaN,NaN,71.696648,72.400513,False
3,71.928040,72.533080,71.708680,72.277563,108872000,NaN,NaN,72.267921,71.696648,False
4,73.085121,73.386438,71.631567,71.631567,132079200,NaN,NaN,71.928040,72.267921,False
